<a href="https://colab.research.google.com/github/NataKrj/Automated-Risk-Scoring-System/blob/main/4_class_balance/Baseline_smote_adasyn_rose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- imports ---
from pathlib import Path
from typing import List, Tuple, Optional
from urllib.parse import urlparse

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE, ADASYN

# --- small utils ---
def is_url(path_or_url: str) -> bool:
    try:
        u = urlparse(path_or_url)
        return u.scheme in {"http", "https"}
    except Exception:
        return False

def name_from_input(path_or_url: str) -> str:
    if is_url(path_or_url):
        p = urlparse(path_or_url).path.rstrip('/')
        stem = Path(p).name
        return Path(stem).stem or "input"
    return Path(path_or_url).stem

# --- split ---
def time_aware_split(df: pd.DataFrame, time_col: Optional[str], test_size: float) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if time_col and time_col in df.columns:
        # ensure proper temporal ordering
        try:
            dff = df.copy()
            dff[time_col] = pd.to_datetime(dff[time_col], errors="coerce")
            dff = dff.sort_values(time_col)
        except Exception:
            dff = df.sort_values(time_col)
        dff = dff.reset_index(drop=True)
    else:
        dff = df.reset_index(drop=True)
    n = len(dff)
    n_test = int(round(n * test_size))
    n_test = min(max(n_test, 1), n - 1)
    return dff.iloc[: n - n_test].copy(), dff.iloc[n - n_test :].copy()

# --- optional TF-IDF ---
def tfidf_fit_transform(train_text: pd.Series, test_text: pd.Series, max_features: int):
    tfidf = TfidfVectorizer(max_features=max_features, stop_words='english', ngram_range=(1,2), min_df=3)
    Xt = tfidf.fit_transform(train_text.fillna(''))
    Xv = tfidf.transform(test_text.fillna(''))
    cols = [f'desc_tfidf_{i}' for i in range(Xt.shape[1])]
    return (pd.DataFrame(Xt.toarray(), columns=cols, index=train_text.index),
            pd.DataFrame(Xv.toarray(), columns=cols, index=test_text.index),
            tfidf)

# --- preprocessing ---
def build_ct(cat_cols: List[str], num_cols: List[str]) -> ColumnTransformer:
    transformers = []
    if cat_cols:
        try:
            ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        except TypeError:
            ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
        transformers.append(("cat", ohe, cat_cols))
    if num_cols:
        transformers.append(("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), num_cols))
    return ColumnTransformer(transformers, remainder="drop")

def apply_ct(ct: ColumnTransformer, Xtr: pd.DataFrame, Xte: pd.DataFrame):
    Xtr_t = ct.fit_transform(Xtr)
    Xte_t = ct.transform(Xte)
    try:
        names = list(map(str, ct.get_feature_names_out()))
    except Exception:
        names = [f"f_{i}" for i in range(Xtr_t.shape[1])]
    num_mask = np.array([n.startswith("num__") for n in names], dtype=bool)
    cat_mask = np.array([n.startswith("cat__") for n in names], dtype=bool)
    return Xtr_t, Xte_t, names, num_mask, cat_mask

# --- ROSE-like ---
def rose_like_resample(X: np.ndarray, y: np.ndarray, num_mask: np.ndarray, target_ratio: float,
                       rng: np.random.Generator, jitter: float = 0.05):
    y = y.copy()
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos, n_neg = len(pos_idx), len(neg_idx)
    cur_ratio = n_pos / max(1, (n_pos + n_neg))
    if target_ratio <= cur_ratio:
        return X, y
    n_pos_target = int(np.ceil(target_ratio / (1.0 - target_ratio) * n_neg))
    G = n_pos_target - n_pos
    if G <= 0:
        return X, y
    sel = rng.choice(pos_idx, size=G, replace=True)
    X_syn = X[sel].copy()
    if num_mask.any():
        noise = rng.normal(0.0, jitter, size=(G, int(num_mask.sum())))
        X_syn[:, num_mask] += noise
    y_syn = np.ones(G, dtype=y.dtype)
    return np.vstack([X, X_syn]), np.concatenate([y, y_syn])

# --- core per-file ---
def process_one(input_ref: str, out_root: Path, label_col: str, time_col: Optional[str], test_size: float,
                sampling_ratio: float, smote_k: int, adasyn_k: int, rose_jitter: float, random_state: int,
                categoricals: Optional[List[str]] = None, text_col: Optional[str] = None,
                text_max_features: int = 100, zip_outputs: bool = False) -> None:
    rng = np.random.default_rng(random_state)

    df = pd.read_csv(input_ref)
    assert label_col in df.columns, f"Label column '{label_col}' not found in {input_ref}"

    # split
    train_df, test_df = time_aware_split(df, time_col, test_size)

    # optional: drop time from features
    if time_col and time_col in train_df.columns:
        train_df = train_df.drop(columns=[time_col])
        test_df  = test_df.drop(columns=[time_col])

    # optional: drop very high-cardinality text/ids if present
    for c in ["Sender", "Receiver", "Description", "Transaction_id"]:
        if c in train_df.columns:
            train_df = train_df.drop(columns=[c])
            test_df  = test_df.drop(columns=[c])

    # optional text features
    if text_col and text_col in train_df.columns:
        Xtr_txt, Xte_txt, _ = tfidf_fit_transform(train_df[text_col], test_df[text_col], text_max_features)
        train_df = pd.concat([train_df.drop(columns=[text_col]), Xtr_txt], axis=1)
        test_df  = pd.concat([test_df.drop(columns=[text_col]),  Xte_txt], axis=1)

    # y and X
    y_train = train_df[label_col].astype(int).to_numpy()
    y_test  = test_df[label_col].astype(int).to_numpy()

    # pick categoricals / numerics
    explicit_cats = set(categoricals or [])
    auto_cats = {c for c in train_df.columns if c != label_col and (not pd.api.types.is_numeric_dtype(train_df[c]))}
    cat_cols = list((explicit_cats | auto_cats) - {label_col})
    num_cols = [c for c in train_df.columns if c not in cat_cols and c != label_col]

    ct = build_ct(cat_cols, num_cols)
    X_train, X_test, feat_names, num_mask, _ = apply_ct(ct, train_df.drop(columns=[label_col]),
                                                        test_df.drop(columns=[label_col]))

    # variants
    X_base,  y_base    = X_train, y_train
    smote              = SMOTE(sampling_strategy=sampling_ratio, k_neighbors=smote_k, random_state=random_state)
    X_smote, y_smote   = smote.fit_resample(X_train, y_train)
    adasyn             = ADASYN(sampling_strategy=sampling_ratio, n_neighbors=adasyn_k, random_state=random_state)
    X_adasyn, y_adasyn = adasyn.fit_resample(X_train, y_train)
    X_rose,  y_rose    = rose_like_resample(X_train, y_train, num_mask=num_mask, target_ratio=sampling_ratio,
                                            rng=rng, jitter=rose_jitter)

    # save
    batch_name = name_from_input(input_ref)
    outdir = out_root / batch_name
    outdir.mkdir(parents=True, exist_ok=True)

    def save_pair(X: np.ndarray, y: np.ndarray, tag: str):
        df_out = pd.DataFrame(X, columns=feat_names)
        df_out[label_col] = y
        df_out.to_csv(outdir / f"train_{tag}.csv", index=False)

    save_pair(X_base,   y_base,   "baseline")
    save_pair(X_smote,  y_smote,  "smote")
    save_pair(X_adasyn, y_adasyn, "adasyn")
    save_pair(X_rose,   y_rose,   "rose")

    df_test = pd.DataFrame(X_test, columns=feat_names)
    df_test[label_col] = y_test
    df_test.to_csv(outdir / "test.csv", index=False)

    def stats(y):
        n1 = int((y==1).sum()); n0 = int((y==0).sum())
        return n1, n0, round(n1/(n1+n0+1e-9), 4)

    pd.DataFrame({
        "variant": ["baseline","smote","adasyn","rose"],
        "pos,neg,frac": [str(stats(y_base)), str(stats(y_smote)), str(stats(y_adasyn)), str(stats(y_rose))]
    }).to_csv(outdir / "_balance_report.csv", index=False)

    if zip_outputs:
        import shutil
        shutil.make_archive(str(outdir), "zip", root_dir=outdir)

    print(f"[OK] {batch_name}: saved train_{{baseline,smote,adasyn,rose}}.csv, test.csv → {outdir}")

# --- batch API ---
def run_batch(*, inputs: List[str], output_dir: str, label_col: str, time_col: Optional[str] = None,
              test_size: float = 0.2, sampling_ratio: float = 0.5, smote_k: int = 5, adasyn_k: int = 5,
              rose_jitter: float = 0.05, random_state: int = 42, categoricals: Optional[List[str]] = None,
              text_col: Optional[str] = None, text_max_features: int = 500, zip_outputs: bool = False) -> None:
    out_root = Path(output_dir)
    out_root.mkdir(parents=True, exist_ok=True)
    for inp in inputs:
        process_one(
            input_ref=inp,
            out_root=out_root,
            label_col=label_col,
            time_col=time_col,
            test_size=test_size,
            sampling_ratio=sampling_ratio,
            smote_k=smote_k,
            adasyn_k=adasyn_k,
            rose_jitter=rose_jitter,
            random_state=random_state,
            categoricals=categoricals,
            text_col=text_col,
            text_max_features=text_max_features,
            zip_outputs=zip_outputs,
        )

In [ ]:
RAW_PREFIX = (
    "https://raw.githubusercontent.com/"
    "NataKrj/Automated-Risk-Scoring-System/"
    "main/1_data_synthesis/Rules_based_synthetic_data/500/"
)

INPUTS = [
    f"{RAW_PREFIX}synthetic_transactions_structured_500_{i}.csv"
    for i in range(1, 6)
]

# Check one file
df = pd.read_csv(INPUTS[0])
print(df.columns.tolist())

run_batch(
    inputs=INPUTS,
    output_dir="./out_balanced",
    label_col="Is_Suspicious",
    time_col="Date",
    test_size=0.2,
    sampling_ratio=0.3,
    smote_k=3, adasyn_k=3,
    categoricals=["Currency","Transaction Type","Transaction Flow","Country"],
    text_col=None,
    zip_outputs=False
)

[OK] synthetic_transactions_structured_100_1: saved train_{baseline,smote,adasyn,rose}.csv, test.csv → out_balanced/synthetic_transactions_structured_100_1
[OK] synthetic_transactions_structured_100_2: saved train_{baseline,smote,adasyn,rose}.csv, test.csv → out_balanced/synthetic_transactions_structured_100_2
[OK] synthetic_transactions_structured_100_3: saved train_{baseline,smote,adasyn,rose}.csv, test.csv → out_balanced/synthetic_transactions_structured_100_3
[OK] synthetic_transactions_structured_100_4: saved train_{baseline,smote,adasyn,rose}.csv, test.csv → out_balanced/synthetic_transactions_structured_100_4
[OK] synthetic_transactions_structured_100_5: saved train_{baseline,smote,adasyn,rose}.csv, test.csv → out_balanced/synthetic_transactions_structured_100_5


In [ ]:
from pathlib import Path
import pandas as pd

root = Path("./out_balanced")
for d in sorted(root.iterdir()):
    print("Batch:", d.name)
    print(pd.read_csv(d / "_balance_report.csv").to_string(index=False), "\n")

Batch: synthetic_transactions_structured_100_1
 variant         pos,neg,frac
baseline    (65, 8015, 0.008)
   smote (2404, 8015, 0.2307)
  adasyn (2398, 8015, 0.2303)
    rose (3436, 8015, 0.3001) 

Batch: synthetic_transactions_structured_100_2
 variant         pos,neg,frac
baseline   (72, 8008, 0.0089)
   smote (2402, 8008, 0.2307)
  adasyn (2402, 8008, 0.2307)
    rose (3433, 8008, 0.3001) 

Batch: synthetic_transactions_structured_100_3
 variant         pos,neg,frac
baseline   (80, 8000, 0.0099)
   smote (2400, 8000, 0.2308)
  adasyn (2401, 8000, 0.2308)
    rose    (3429, 8000, 0.3) 

Batch: synthetic_transactions_structured_100_4
 variant         pos,neg,frac
baseline   (71, 8009, 0.0088)
   smote (2402, 8009, 0.2307)
  adasyn (2410, 8009, 0.2313)
    rose    (3433, 8009, 0.3) 

Batch: synthetic_transactions_structured_100_5
 variant         pos,neg,frac
baseline   (70, 8010, 0.0087)
   smote (2403, 8010, 0.2308)
  adasyn (2403, 8010, 0.2308)
    rose    (3433, 8010, 0.3) 



In [ ]:
import ssl
import pandas as pd

ssl._create_default_https_context = ssl._create_unverified_context

# BASELINE (no resampling)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.parse import urlparse
from typing import List, Dict, Any

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# --- CONFIG ---
RAW_PREFIX = (
    "https://raw.githubusercontent.com/"
    "NataKrj/Automated-Risk-Scoring-System/"
    "main/1_data_synthesis/Rules_based_synthetic_data/500/"
)
INPUTS: List[str] = [f"{RAW_PREFIX}synthetic_transactions_structured_500_{i}.csv" for i in range(1, 6)]
OUTPUT_ROOT = Path("./baseline_500")
LABEL_COL = "Is_Suspicious"
TIME_COL  = "Date"
TEST_SIZE = 0.20
TFIDF_MAX = 100
DROP_ALWAYS = ["Transaction_id", "Suspicion Category"]

CATEGORICAL_RAW = [
    "Sender", "Receiver", "Currency", "Country", "Transaction Type", "Transaction Flow"
]

# --- helpers ---

def _name_from_ref(s: str) -> str:
    p = urlparse(s)
    name = Path(p.path).name if p.scheme in {"http","https"} else Path(s).name
    return Path(name).stem


def _time_split(df: pd.DataFrame, time_col: str, test_frac: float):
    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df); n_test = max(1, int(round(n * test_frac)))
    return df.iloc[: n - n_test].copy(), df.iloc[n - n_test :].copy()


def _fit_maps(train: pd.DataFrame) -> Dict[str, Any]:
    """Build train-only maps (no leakage): frequencies, amount aggregates, TF‑IDF."""
    maps: Dict[str, Any] = {}
    # frequency maps for all raw categoricals
    for col in CATEGORICAL_RAW:
        vc = train[col].value_counts(dropna=False)
        maps[f"freq_{col}"] = vc
    # sender/receiver numeric aggregates
    maps["sender_avg_amount"]   = train.groupby("Sender")["Amount"].mean()
    maps["receiver_avg_amount"] = train.groupby("Receiver")["Amount"].mean()
    maps["global_amount_mean"]  = float(train["Amount"].mean())
    qs = np.linspace(0, 1, 6)
    maps["amount_bins"] = np.quantile(train["Amount"], qs)
    # TF-IDF on Description
    tfidf = TfidfVectorizer(max_features=TFIDF_MAX, stop_words="english", ngram_range=(1,2), min_df=3)
    tfidf.fit(train["Description"].fillna(""))
    maps["tfidf"] = tfidf
    return maps


def _transform(df: pd.DataFrame, maps: Dict[str, Any]) -> pd.DataFrame:
    """Apply maps to any split (train/test). Return numeric feature table."""
    d = df.copy()
    # date features
    d["Date"] = pd.to_datetime(d["Date"])
    d["dow"] = d["Date"].dt.dayofweek
    d["mon"] = d["Date"].dt.month
    d["is_weekend"] = d["dow"].isin([5,6]).astype(int)
    # cyclical encoding
    d["dow_sin"], d["dow_cos"] = np.sin(2*np.pi*d["dow"]/7),  np.cos(2*np.pi*d["dow"]/7)
    d["mon_sin"], d["mon_cos"] = np.sin(2*np.pi*(d["mon"]-1)/12), np.cos(2*np.pi*(d["mon"]-1)/12)

    # TF-IDF
    tfidf = maps["tfidf"]
    Xt = tfidf.transform(d["Description"].fillna(""))
    tf_cols = [f"desc_tfidf_{i}" for i in range(Xt.shape[1])]
    Xtf = pd.DataFrame(Xt.toarray(), columns=tf_cols, index=d.index)

    # simple text flags
    d["desc_length"] = d["Description"].fillna("").str.len()
    d["has_invoice"] = d["Description"].fillna("").str.contains("invoice|inv", case=False).astype(int)

    # frequency encodings
    for col in CATEGORICAL_RAW:
        vc = maps[f"freq_{col}"]
        d[f"{col}_freq"] = d[col].map(vc).fillna(0).astype(float)

    # sender/receiver aggregates and deviations
    s_mu = maps["sender_avg_amount"]
    r_mu = maps["receiver_avg_amount"]
    g_mu = maps["global_amount_mean"]
    d["sender_avg_amount"]   = d["Sender"].map(s_mu).fillna(g_mu)
    d["receiver_avg_amount"] = d["Receiver"].map(r_mu).fillna(g_mu)
    d["amount_log"] = np.log1p(d["Amount"].astype(float))
    d["amount_deviation"] = (d["Amount"].astype(float) - d["sender_avg_amount"]).abs()

    # amount bin
    bin_edges = maps["amount_bins"]
    d["amount_bin"] = np.clip(np.digitize(d["Amount"], bin_edges, right=True) - 1, 0, len(bin_edges)-2)

    # assemble numeric features
    keep_cols = [
        "dow","mon","is_weekend","dow_sin","dow_cos","mon_sin","mon_cos",
        "desc_length","has_invoice",
        "Sender_freq","Receiver_freq","Currency_freq","Country_freq","Transaction Type_freq","Transaction Flow_freq",
        "sender_avg_amount","receiver_avg_amount","amount_log","amount_deviation","amount_bin",
    ]
    Xnum = d[[c for c in keep_cols if c in d.columns]].reset_index(drop=True)
    X = pd.concat([Xnum, Xtf.reset_index(drop=True)], axis=1)
    return X

# --- run ---
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for ref in INPUTS:
    df = pd.read_csv(ref)
    assert LABEL_COL in df.columns, f"Label '{LABEL_COL}' not found in {ref}"
    train_df, test_df = _time_split(df.drop(columns=[c for c in DROP_ALWAYS if c in df.columns]), TIME_COL, TEST_SIZE)

    # build maps on train only
    maps = _fit_maps(train_df)
    Xtr = _transform(train_df, maps)
    Xte = _transform(test_df,  maps)

    y_tr = train_df[LABEL_COL].astype(int).to_numpy()
    y_te = test_df[LABEL_COL].astype(int).to_numpy()

    # scale
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)

    # save
    outdir = OUTPUT_ROOT / _name_from_ref(ref)
    outdir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(Xtr_s).assign(**{LABEL_COL: y_tr}).to_csv(outdir/"train_baseline.csv", index=False)
    pd.DataFrame(Xte_s).assign(**{LABEL_COL: y_te}).to_csv(outdir/"test.csv", index=False)

    def _stats(y):
        p = int((y==1).sum()); n = int((y==0).sum());
        return p, n, round(p/(p+n+1e-9), 4)
    pd.DataFrame({"variant":["baseline"], "pos,neg,frac":[str(_stats(y_tr))]}).to_csv(outdir/"_balance_report.csv", index=False)
    print(f"[baseline FE OK] → {outdir}")

[baseline FE OK] → baseline_500/synthetic_transactions_structured_500_1
[baseline FE OK] → baseline_500/synthetic_transactions_structured_500_2
[baseline FE OK] → baseline_500/synthetic_transactions_structured_500_3
[baseline FE OK] → baseline_500/synthetic_transactions_structured_500_4
[baseline FE OK] → baseline_500/synthetic_transactions_structured_500_5


# SMOTE

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.parse import urlparse
from typing import List, Dict, Any
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE

# --- CONFIG ---
RAW_PREFIX = (
    "https://raw.githubusercontent.com/"
    "NataKrj/Automated-Risk-Scoring-System/"
    "main/1_data_synthesis/Rules_based_synthetic_data/500/"
)
INPUTS: List[str] = [f"{RAW_PREFIX}synthetic_transactions_structured_500_{i}.csv" for i in range(1, 6)]
OUTPUT_ROOT = Path("./smote_500")
LABEL_COL = "Is_Suspicious"; TIME_COL = "Date"; TEST_SIZE = 0.20
TFIDF_MAX = 100
DROP_ALWAYS = ["Transaction_id", "Suspicion Category"]
SAMPLING_RATIO = 0.5
SMOTE_K = 5

CATEGORICAL_RAW = ["Sender","Receiver","Currency","Country","Transaction Type","Transaction Flow"]

from urllib.parse import urlparse

def _name_from_ref(s: str) -> str:
    p = urlparse(s)
    name = Path(p.path).name if p.scheme in {"http","https"} else Path(s).name
    return Path(name).stem

def _time_split(df: pd.DataFrame, time_col: str, test_frac: float):
    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df); n_test = max(1, int(round(n * test_frac)))
    return df.iloc[: n - n_test].copy(), df.iloc[n - n_test :].copy()

def _fit_maps(train: pd.DataFrame):
    maps = {}
    for col in CATEGORICAL_RAW:
        maps[f"freq_{col}"] = train[col].value_counts(dropna=False)
    maps["sender_avg_amount"] = train.groupby("Sender")["Amount"].mean()
    maps["receiver_avg_amount"] = train.groupby("Receiver")["Amount"].mean()
    maps["global_amount_mean"] = float(train["Amount"].mean())
    maps["amount_bins"] = np.quantile(train["Amount"], np.linspace(0,1,6))
    tfidf = TfidfVectorizer(max_features=TFIDF_MAX, stop_words="english", ngram_range=(1,2), min_df=3)
    tfidf.fit(train["Description"].fillna(""))
    maps["tfidf"] = tfidf
    return maps

def _transform(df: pd.DataFrame, maps):
    d = df.copy()
    d["Date"] = pd.to_datetime(d["Date"])
    d["dow"] = d["Date"].dt.dayofweek
    d["mon"] = d["Date"].dt.month
    d["is_weekend"] = d["dow"].isin([5,6]).astype(int)
    d["dow_sin"], d["dow_cos"] = np.sin(2*np.pi*d["dow"]/7),  np.cos(2*np.pi*d["dow"]/7)
    d["mon_sin"], d["mon_cos"] = np.sin(2*np.pi*(d["mon"]-1)/12), np.cos(2*np.pi*(d["mon"]-1)/12)

    tfidf = maps["tfidf"]; Xt = tfidf.transform(d["Description"].fillna(""))
    tf_cols = [f"desc_tfidf_{i}" for i in range(Xt.shape[1])]
    Xtf = pd.DataFrame(Xt.toarray(), columns=tf_cols, index=d.index)

    d["desc_length"] = d["Description"].fillna("").str.len()
    d["has_invoice"] = d["Description"].fillna("").str.contains("invoice|inv", case=False).astype(int)

    for col in CATEGORICAL_RAW:
        d[f"{col}_freq"] = d[col].map(maps[f"freq_{col}"]).fillna(0).astype(float)

    s_mu = maps["sender_avg_amount"]; r_mu = maps["receiver_avg_amount"]; g_mu = maps["global_amount_mean"]
    d["sender_avg_amount"] = d["Sender"].map(s_mu).fillna(g_mu)
    d["receiver_avg_amount"] = d["Receiver"].map(r_mu).fillna(g_mu)
    d["amount_log"] = np.log1p(d["Amount"].astype(float))
    d["amount_deviation"] = (d["Amount"].astype(float) - d["sender_avg_amount"]).abs()

    bin_edges = maps["amount_bins"]
    d["amount_bin"] = np.clip(np.digitize(d["Amount"], bin_edges, right=True) - 1, 0, len(bin_edges)-2)

    keep_cols = [
        "dow","mon","is_weekend","dow_sin","dow_cos","mon_sin","mon_cos",
        "desc_length","has_invoice",
        "Sender_freq","Receiver_freq","Currency_freq","Country_freq","Transaction Type_freq","Transaction Flow_freq",
        "sender_avg_amount","receiver_avg_amount","amount_log","amount_deviation","amount_bin",
    ]
    Xnum = d[[c for c in keep_cols if c in d.columns]].reset_index(drop=True)
    X = pd.concat([Xnum, Xtf.reset_index(drop=True)], axis=1)
    return X

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for ref in INPUTS:
    df = pd.read_csv(ref)
    assert LABEL_COL in df.columns
    train_df, test_df = _time_split(df.drop(columns=[c for c in DROP_ALWAYS if c in df.columns]), TIME_COL, TEST_SIZE)
    maps = _fit_maps(train_df)
    Xtr = _transform(train_df, maps)
    Xte = _transform(test_df, maps)
    y_tr = train_df[LABEL_COL].astype(int).to_numpy()
    y_te = test_df[LABEL_COL].astype(int).to_numpy()

    scaler = StandardScaler(); Xtr_s = scaler.fit_transform(Xtr); Xte_s = scaler.transform(Xte)

    smote = SMOTE(sampling_strategy=SAMPLING_RATIO, k_neighbors=SMOTE_K, random_state=42)
    Xtr_bal, ytr_bal = smote.fit_resample(Xtr_s, y_tr)

    outdir = OUTPUT_ROOT / _name_from_ref(ref); outdir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(Xtr_bal).assign(**{LABEL_COL: ytr_bal}).to_csv(outdir/"train_smote.csv", index=False)
    pd.DataFrame(Xte_s).assign(**{LABEL_COL: y_te}).to_csv(outdir/"test.csv", index=False)

    def _stats(y):
        p = int((y==1).sum()); n = int((y==0).sum());
        return p, n, round(p/(p+n+1e-9), 4)
    pd.DataFrame({"variant":["smote"], "pos,neg,frac":[str(_stats(ytr_bal))]}).to_csv(outdir/"_balance_report.csv", index=False)
    print(f"[smote FE OK] → {outdir}")

[smote FE OK] → smote_500/synthetic_transactions_structured_500_1
[smote FE OK] → smote_500/synthetic_transactions_structured_500_2
[smote FE OK] → smote_500/synthetic_transactions_structured_500_3
[smote FE OK] → smote_500/synthetic_transactions_structured_500_4
[smote FE OK] → smote_500/synthetic_transactions_structured_500_5


# ADASYN

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.parse import urlparse
from typing import List, Dict, Any
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import ADASYN

RAW_PREFIX = (
    "https://raw.githubusercontent.com/"
    "NataKrj/Automated-Risk-Scoring-System/"
    "main/1_data_synthesis/Rules_based_synthetic_data/500/"
)
INPUTS: List[str] = [f"{RAW_PREFIX}synthetic_transactions_structured_500_{i}.csv" for i in range(1, 6)]
OUTPUT_ROOT = Path("./adasyn_500")
LABEL_COL = "Is_Suspicious"; TIME_COL = "Date"; TEST_SIZE = 0.20
TFIDF_MAX = 100
DROP_ALWAYS = ["Transaction_id", "Suspicion Category"]
SAMPLING_RATIO = 0.5
ADASYN_K = 5

CATEGORICAL_RAW = ["Sender","Receiver","Currency","Country","Transaction Type","Transaction Flow"]

from urllib.parse import urlparse

def _name_from_ref(s: str) -> str:
    p = urlparse(s)
    name = Path(p.path).name if p.scheme in {"http","https"} else Path(s).name
    return Path(name).stem

def _time_split(df: pd.DataFrame, time_col: str, test_frac: float):
    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df); n_test = max(1, int(round(n * test_frac)))
    return df.iloc[: n - n_test].copy(), df.iloc[n - n_test :].copy()

def _fit_maps(train: pd.DataFrame):
    maps = {}
    for col in CATEGORICAL_RAW:
        maps[f"freq_{col}"] = train[col].value_counts(dropna=False)
    maps["sender_avg_amount"] = train.groupby("Sender")["Amount"].mean()
    maps["receiver_avg_amount"] = train.groupby("Receiver")["Amount"].mean()
    maps["global_amount_mean"] = float(train["Amount"].mean())
    maps["amount_bins"] = np.quantile(train["Amount"], np.linspace(0,1,6))
    tfidf = TfidfVectorizer(max_features=TFIDF_MAX, stop_words="english", ngram_range=(1,2), min_df=3)
    tfidf.fit(train["Description"].fillna(""))
    maps["tfidf"] = tfidf
    return maps

def _transform(df: pd.DataFrame, maps):
    d = df.copy()
    d["Date"] = pd.to_datetime(d["Date"])
    d["dow"] = d["Date"].dt.dayofweek
    d["mon"] = d["Date"].dt.month
    d["is_weekend"] = d["dow"].isin([5,6]).astype(int)
    d["dow_sin"], d["dow_cos"] = np.sin(2*np.pi*d["dow"]/7),  np.cos(2*np.pi*d["dow"]/7)
    d["mon_sin"], d["mon_cos"] = np.sin(2*np.pi*(d["mon"]-1)/12), np.cos(2*np.pi*(d["mon"]-1)/12)

    tfidf = maps["tfidf"]; Xt = tfidf.transform(d["Description"].fillna(""))
    tf_cols = [f"desc_tfidf_{i}" for i in range(Xt.shape[1])]
    Xtf = pd.DataFrame(Xt.toarray(), columns=tf_cols, index=d.index)

    d["desc_length"] = d["Description"].fillna("").str.len()
    d["has_invoice"] = d["Description"].fillna("").str.contains("invoice|inv", case=False).astype(int)

    for col in CATEGORICAL_RAW:
        d[f"{col}_freq"] = d[col].map(maps[f"freq_{col}"]).fillna(0).astype(float)

    s_mu = maps["sender_avg_amount"]; r_mu = maps["receiver_avg_amount"]; g_mu = maps["global_amount_mean"]
    d["sender_avg_amount"] = d["Sender"].map(s_mu).fillna(g_mu)
    d["receiver_avg_amount"] = d["Receiver"].map(r_mu).fillna(g_mu)
    d["amount_log"] = np.log1p(d["Amount"].astype(float))
    d["amount_deviation"] = (d["Amount"].astype(float) - d["sender_avg_amount"]).abs()

    bin_edges = maps["amount_bins"]
    d["amount_bin"] = np.clip(np.digitize(d["Amount"], bin_edges, right=True) - 1, 0, len(bin_edges)-2)

    keep_cols = [
        "dow","mon","is_weekend","dow_sin","dow_cos","mon_sin","mon_cos",
        "desc_length","has_invoice",
        "Sender_freq","Receiver_freq","Currency_freq","Country_freq","Transaction Type_freq","Transaction Flow_freq",
        "sender_avg_amount","receiver_avg_amount","amount_log","amount_deviation","amount_bin",
    ]
    Xnum = d[[c for c in keep_cols if c in d.columns]].reset_index(drop=True)
    X = pd.concat([Xnum, Xtf.reset_index(drop=True)], axis=1)
    return X

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for ref in INPUTS:
    df = pd.read_csv(ref)
    assert LABEL_COL in df.columns
    train_df, test_df = _time_split(df.drop(columns=[c for c in DROP_ALWAYS if c in df.columns]), TIME_COL, TEST_SIZE)
    maps = _fit_maps(train_df)
    Xtr = _transform(train_df, maps); Xte = _transform(test_df, maps)
    y_tr = train_df[LABEL_COL].astype(int).to_numpy(); y_te = test_df[LABEL_COL].astype(int).to_numpy()

    scaler = StandardScaler(); Xtr_s = scaler.fit_transform(Xtr); Xte_s = scaler.transform(Xte)

    ada = ADASYN(sampling_strategy=SAMPLING_RATIO, n_neighbors=ADASYN_K, random_state=42)
    Xtr_bal, ytr_bal = ada.fit_resample(Xtr_s, y_tr)

    outdir = OUTPUT_ROOT / _name_from_ref(ref); outdir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(Xtr_bal).assign(**{LABEL_COL: ytr_bal}).to_csv(outdir/"train_adasyn.csv", index=False)
    pd.DataFrame(Xte_s).assign(**{LABEL_COL: y_te}).to_csv(outdir/"test.csv", index=False)

    def _stats(y):
        p = int((y==1).sum()); n = int((y==0).sum());
        return p, n, round(p/(p+n+1e-9), 4)
    pd.DataFrame({"variant":["adasyn"], "pos,neg,frac":[str(_stats(ytr_bal))]}).to_csv(outdir/"_balance_report.csv", index=False)
    print(f"[adasyn FE OK] → {outdir}")

[adasyn FE OK] → adasyn_500/synthetic_transactions_structured_500_1
[adasyn FE OK] → adasyn_500/synthetic_transactions_structured_500_2
[adasyn FE OK] → adasyn_500/synthetic_transactions_structured_500_3
[adasyn FE OK] → adasyn_500/synthetic_transactions_structured_500_4
[adasyn FE OK] → adasyn_500/synthetic_transactions_structured_500_5


# ROSE

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from urllib.parse import urlparse
from typing import List, Dict, Any
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

RAW_PREFIX = (
    "https://raw.githubusercontent.com/"
    "NataKrj/Automated-Risk-Scoring-System/"
    "main/1_data_synthesis/Rules_based_synthetic_data/500/"
)
INPUTS: List[str] = [f"{RAW_PREFIX}synthetic_transactions_structured_500_{i}.csv" for i in range(1, 6)]
OUTPUT_ROOT = Path("./rose_500")
LABEL_COL = "Is_Suspicious"; TIME_COL = "Date"; TEST_SIZE = 0.20
TFIDF_MAX = 100
DROP_ALWAYS = ["Transaction_id", "Suspicion Category"]
SAMPLING_RATIO = 0.5
ROSE_JITTER = 0.05

CATEGORICAL_RAW = ["Sender","Receiver","Currency","Country","Transaction Type","Transaction Flow"]

from urllib.parse import urlparse

def _name_from_ref(s: str) -> str:
    p = urlparse(s)
    name = Path(p.path).name if p.scheme in {"http","https"} else Path(s).name
    return Path(name).stem

def _time_split(df: pd.DataFrame, time_col: str, test_frac: float):
    if time_col in df.columns:
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).reset_index(drop=True)
    n = len(df); n_test = max(1, int(round(n * test_frac)))
    return df.iloc[: n - n_test].copy(), df.iloc[n - n_test :].copy()

def _fit_maps(train: pd.DataFrame):
    maps = {}
    for col in CATEGORICAL_RAW:
        maps[f"freq_{col}"] = train[col].value_counts(dropna=False)
    maps["sender_avg_amount"] = train.groupby("Sender")["Amount"].mean()
    maps["receiver_avg_amount"] = train.groupby("Receiver")["Amount"].mean()
    maps["global_amount_mean"] = float(train["Amount"].mean())
    maps["amount_bins"] = np.quantile(train["Amount"], np.linspace(0,1,6))
    tfidf = TfidfVectorizer(max_features=TFIDF_MAX, stop_words="english", ngram_range=(1,2), min_df=3)
    tfidf.fit(train["Description"].fillna(""))
    maps["tfidf"] = tfidf
    return maps

def _transform(df: pd.DataFrame, maps):
    d = df.copy()
    d["Date"] = pd.to_datetime(d["Date"])
    d["dow"] = d["Date"].dt.dayofweek
    d["mon"] = d["Date"].dt.month
    d["is_weekend"] = d["dow"].isin([5,6]).astype(int)
    d["dow_sin"], d["dow_cos"] = np.sin(2*np.pi*d["dow"]/7),  np.cos(2*np.pi*d["dow"]/7)
    d["mon_sin"], d["mon_cos"] = np.sin(2*np.pi*(d["mon"]-1)/12), np.cos(2*np.pi*(d["mon"]-1)/12)

    tfidf = maps["tfidf"]; Xt = tfidf.transform(d["Description"].fillna(""))
    tf_cols = [f"desc_tfidf_{i}" for i in range(Xt.shape[1])]
    Xtf = pd.DataFrame(Xt.toarray(), columns=tf_cols, index=d.index)

    d["desc_length"] = d["Description"].fillna("").str.len()
    d["has_invoice"] = d["Description"].fillna("").str.contains("invoice|inv", case=False).astype(int)

    for col in CATEGORICAL_RAW:
        d[f"{col}_freq"] = d[col].map(maps[f"freq_{col}"]).fillna(0).astype(float)

    s_mu = maps["sender_avg_amount"]; r_mu = maps["receiver_avg_amount"]; g_mu = maps["global_amount_mean"]
    d["sender_avg_amount"] = d["Sender"].map(s_mu).fillna(g_mu)
    d["receiver_avg_amount"] = d["Receiver"].map(r_mu).fillna(g_mu)
    d["amount_log"] = np.log1p(d["Amount"].astype(float))
    d["amount_deviation"] = (d["Amount"].astype(float) - d["sender_avg_amount"]).abs()

    bin_edges = maps["amount_bins"]
    d["amount_bin"] = np.clip(np.digitize(d["Amount"], bin_edges, right=True) - 1, 0, len(bin_edges)-2)

    keep_cols = [
        "dow","mon","is_weekend","dow_sin","dow_cos","mon_sin","mon_cos",
        "desc_length","has_invoice",
        "Sender_freq","Receiver_freq","Currency_freq","Country_freq","Transaction Type_freq","Transaction Flow_freq",
        "sender_avg_amount","receiver_avg_amount","amount_log","amount_deviation","amount_bin",
    ]
    Xnum = d[[c for c in keep_cols if c in d.columns]].reset_index(drop=True)
    X = pd.concat([Xnum, Xtf.reset_index(drop=True)], axis=1)
    return X

# ROSE
def _rose_like_resample(X: np.ndarray, y: np.ndarray, target_ratio: float, jitter: float, rng: np.random.Generator):
    y = y.copy()
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    n_pos, n_neg = len(pos_idx), len(neg_idx)
    cur_ratio = n_pos / max(1, (n_pos + n_neg))
    if target_ratio <= cur_ratio:
        return X, y
    n_pos_target = int(np.ceil(target_ratio / (1.0 - target_ratio) * n_neg))
    G = n_pos_target - n_pos
    if G <= 0:
        return X, y
    sel = rng.choice(pos_idx, size=G, replace=True)
    X_syn = X[sel].copy()
    noise = rng.normal(0.0, jitter, size=X_syn.shape)  # jitter across standardized space
    X_syn = X_syn + noise
    y_syn = np.ones(G, dtype=y.dtype)
    return np.vstack([X, X_syn]), np.concatenate([y, y_syn])

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for ref in INPUTS:
    df = pd.read_csv(ref)
    assert LABEL_COL in df.columns
    train_df, test_df = _time_split(df.drop(columns=[c for c in DROP_ALWAYS if c in df.columns]), TIME_COL, TEST_SIZE)
    maps = _fit_maps(train_df)
    Xtr = _transform(train_df, maps); Xte = _transform(test_df, maps)
    y_tr = train_df[LABEL_COL].astype(int).to_numpy(); y_te = test_df[LABEL_COL].astype(int).to_numpy()

    scaler = StandardScaler(); Xtr_s = scaler.fit_transform(Xtr); Xte_s = scaler.transform(Xte)

    rng = np.random.default_rng(42)
    Xtr_bal, ytr_bal = _rose_like_resample(Xtr_s, y_tr, target_ratio=SAMPLING_RATIO, jitter=ROSE_JITTER, rng=rng)

    outdir = OUTPUT_ROOT / _name_from_ref(ref); outdir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(Xtr_bal).assign(**{LABEL_COL: ytr_bal}).to_csv(outdir/"train_rose.csv", index=False)
    pd.DataFrame(Xte_s).assign(**{LABEL_COL: y_te}).to_csv(outdir/"test.csv", index=False)

    def _stats(y):
        p = int((y==1).sum()); n = int((y==0).sum());
        return p, n, round(p/(p+n+1e-9), 4)
    pd.DataFrame({"variant":["rose"], "pos,neg,frac":[str(_stats(ytr_bal))]}).to_csv(outdir/"_balance_report.csv", index=False)
    print(f"[rose FE OK] → {outdir}")

[rose FE OK] → rose_500/synthetic_transactions_structured_500_1
[rose FE OK] → rose_500/synthetic_transactions_structured_500_2
[rose FE OK] → rose_500/synthetic_transactions_structured_500_3
[rose FE OK] → rose_500/synthetic_transactions_structured_500_4
[rose FE OK] → rose_500/synthetic_transactions_structured_500_5
